# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 3: Evaluation, Baselines, Traditional ML

Today we'll write some simple models to predict the price of a product

We'll use an approach to evaluate the performance of the model

And we'll test some Baseline Models using Traditional machine learning

In [1]:
import random
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from pricer.evaluator import evaluate
from pricer.items import Item

In [8]:
LITE_MODE = True

In [9]:
username = "jayalexander"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

README.md:   0%|          | 0.00/735 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.05M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/302k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/305k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


In [10]:
def random_pricer(item):
    return random.randrange(1,1000)

In [11]:
random.seed(42)
evaluate(random_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$436 $1 $29 $690 $252 $21 $85 $72 $719 $225 $20 $380 $894 $505 $11 $572 $354 $17 $179 $23 $90 $115 $433 $442 $304 $122 $291 $714 $567 $639 $539 $370 $66 $380 $489 $534 $769 $835 $207 $740 $626 $84 $680 $178 $129 $260 $142 $189 $836 $580 $310 $25 $380 $270 $47 $234 $861 $313 $417 $259 $591 $33 $657 $361 $79 $38 $757 $500 $263 $5 $534 $284 $570 $625 $584 $871 $759 $361 $575 $178 $602 $60 $17 $579 $207 $732 $115 $224 $756 $193 $866 $9 $370 $250 $456 $423 $821 $217 $103 $195 $264 $98 $650 $135 $470 $842 $675 $264 $43 $325 $591 $138 $516 $619 $56 $2 $449 $369 $221 $845 $640 $616 $501 $59 $502 $273 $844 $688 $616 $81 $164 $705 $52 $795 $259 $396 $70 $2 $89 $798 $902 $331 $818 $716 $129 $186 $627 $2 $141 $873 $918 $553 $423 $7 $253 $136 $204 $707 $255 $502 $19 $739 $557 $426 $80 $575 $39 $321 $185 $132 $497 $484 $326 $751 $53 $733 $85 $64 $549 $71 $158 $672 $133 $362 $16 $373 $258 $544 $420 $515 $223 $944 $532 $743 $866 $68 $527 $459 $67 $673 

In [12]:
# That was fun!
# We can do better - here's another rather trivial model

training_prices = [item.price for item in train]
training_average = sum(training_prices) / len(training_prices)
print(training_average)

def constant_pricer(item):
    return training_average

140.347293


In [13]:
evaluate(constant_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$79 $24 $85 $70 $110 $90 $4 $75 $105 $190 $573 $239 $120 $86 $61 $107 $61 $90 $70 $21 $6 $16 $55 $35 $192 $312 $355 $120 $42 $60 $120 $80 $20 $60 $25 $679 $81 $84 $73 $102 $59 $60 $105 $115 $80 $115 $122 $109 $5 $62 $105 $10 $335 $21 $87 $6 $133 $100 $62 $128 $96 $62 $49 $30 $488 $50 $100 $305 $15 $64 $109 $123 $140 $121 $90 $104 $16 $130 $123 $122 $20 $128 $110 $41 $113 $80 $42 $166 $20 $95 $118 $45 $120 $105 $132 $88 $106 $17 $130 $435 $40 $23 $103 $1 $109 $22 $115 $260 $109 $159 $80 $174 $109 $12 $55 $30 $116 $120 $84 $37 $124 $51 $70 $26 $60 $80 $120 $41 $39 $1 $69 $3 $55 $110 $75 $125 $65 $70 $12 $2 $76 $110 $60 $121 $54 $108 $95 $370 $125 $107 $121 $34 $93 $0 $121 $139 $91 $84 $179 $90 $149 $114 $98 $127 $700 $117 $307 $90 $100 $130 $115 $118 $280 $118 $39 $9 $112 $47 $46 $47 $514 $115 $160 $108 $90 $118 $7 $73 $80 $114 $105 $89 $105 $2 $40 $60 $30 $140 $89 $114 

In [14]:
def get_features(item):
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight==0 else 0,
        "text_length": len(item.summary)
    }

In [15]:
def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

In [16]:
# Traditional Linear Regression!

np.random.seed(42)

# Separate features and target
feature_columns = ['weight', 'weight_unknown', 'text_length']

X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

# Train a Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)

for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"Intercept: {model.intercept_}")

# Predict the test set and evaluate
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared Score: {r2}")

weight: 3.5687144025395194
weight_unknown: 20.339946837064403
text_length: 0.21437410927895648
Intercept: 37.55696558415252
Mean Squared Error: 20054.362349806375
R-squared Score: 0.162687326839838


In [17]:
def linear_regression_pricer(item):
    features = get_features(item)
    features_df = pd.DataFrame([features])
    return model.predict(features_df)[0]

In [18]:
evaluate(linear_regression_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$92 $57 $63 $78 $81 $46 $6 $38 $82 $192 $589 $230 $91 $68 $26 $82 $32 $55 $37 $5 $16 $12 $70 $17 $186 $323 $351 $98 $45 $18 $107 $90 $30 $10 $81 $657 $70 $61 $83 $75 $75 $41 $70 $71 $99 $84 $108 $96 $11 $48 $86 $0 $350 $14 $60 $19 $123 $88 $51 $120 $93 $36 $37 $6 $236 $33 $91 $263 $6 $73 $88 $102 $157 $99 $86 $70 $13 $93 $107 $98 $70 $101 $98 $7 $75 $52 $70 $176 $59 $48 $87 $43 $94 $63 $88 $117 $107 $6 $140 $394 $42 $14 $89 $28 $116 $2 $90 $283 $99 $190 $58 $143 $98 $59 $76 $59 $100 $106 $68 $17 $91 $35 $50 $35 $118 $61 $81 $108 $59 $8 $40 $15 $26 $79 $55 $93 $80 $37 $7 $7 $32 $128 $52 $90 $1 $78 $58 $366 $1 $81 $78 $16 $59 $16 $90 $124 $96 $52 $44 $66 $161 $87 $76 $114 $726 $101 $298 $99 $83 $109 $89 $93 $249 $78 $141 $66 $98 $16 $46 $41 $299 $85 $43 $86 $70 $90 $13 $79 $53 $76 $86 $67 $93 $26 $7 $89 $46 $113 $81 $99 

In [19]:
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [20]:
np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(documents)


In [21]:
# Here are the 1,000 most common words that it picked, not including "stop words":

selected_words = vectorizer.get_feature_names_out()
print(f"Number of selected words: {len(selected_words)}")
print("Selected words:", selected_words[1000:1020])

Number of selected words: 2000
Selected words: ['jigsaw' 'joint' 'joints' 'keeping' 'keeps' 'key' 'keyboard' 'keyless'
 'keypad' 'keys' 'kg' 'khz' 'kia' 'kickstand' 'kids' 'king' 'kingston'
 'kit' 'kitchen' 'knife']


In [22]:
regressor = LinearRegression()
regressor.fit(X, prices)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [23]:
def natural_language_linear_regression_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(regressor.predict(x)[0], 0)

In [24]:
evaluate(natural_language_linear_regression_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$146 $133 $55 $35 $251 $205 $92 $4 $92 $135 $515 $160 $141 $134 $121 $125 $21 $58 $77 $61 $45 $15 $8 $0 $133 $254 $7 $42 $110 $19 $186 $34 $56 $7 $122 $320 $1 $55 $116 $91 $96 $143 $14 $65 $177 $5 $79 $3 $74 $32 $42 $34 $169 $4 $41 $198 $119 $148 $59 $19 $9 $44 $6 $6 $380 $22 $123 $181 $76 $163 $28 $20 $101 $116 $18 $29 $100 $8 $150 $33 $19 $65 $30 $16 $40 $67 $31 $90 $112 $56 $77 $83 $117 $2 $98 $62 $8 $17 $219 $231 $145 $34 $37 $42 $3 $9 $24 $128 $31 $4 $6 $17 $115 $9 $38 $18 $172 $20 $63 $93 $59 $86 $7 $68 $100 $36 $40 $45 $19 $113 $4 $31 $93 $30 $19 $83 $82 $73 $34 $106 $39 $134 $32 $107 $234 $7 $24 $163 $3 $9 $16 $240 $63 $110 $20 $43 $79 $24 $78 $34 $142 $11 $11 $23 $455 $19 $115 $50 $31 $10 $15 $36 $316 $102 $84 $13 $28 $54 $41 $29 $412 $104 $65 $117 $139 $20 $86 $67 $150 $21 $51 $69 $16 $20 $27 $81 $33 $53 $12 $26 

In [25]:
subset = 15_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X[:subset], prices[:subset])

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


## Random Forest model

The Random Forest is a type of "**ensemble**" algorithm, meaning that it combines many smaller algorithms to make better predictions.

It uses a very simple kind of machine learning algorithm called a **decision tree**. A decision tree makes predictions by examining the values of features in the input. Like a flow chart with IF statements. Decision trees are very quick and simple, but they tend to overfit.

In our case, the "features" are the elements of the Vector - in other words, it's the number of times that a particular word appears in the product description.

So you can think of it something like this:

**Decision Tree**  
\- IF the word "TV" appears more than 3 times THEN  
-- IF the word "LED" appears more than 2 times THEN  
--- IF the word "HD" appears at least once THEN  
---- Price = $500


With Random Forest, multiple decision trees are created. Each one is trained with a different random subset of the data, and a different random subset of the features. You can see above that we specify 100 trees, which is the default.

Then the Random Forest model simply takes the average of all its trees to product the final result.

In [26]:
def random_forest(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

In [27]:
evaluate(random_forest, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$104 $121 $12 $17 $164 $170 $71 $69 $45 $152 $584 $233 $65 $40 $5 $4 $41 $7 $22 $52 $43 $35 $4 $68 $133 $297 $12 $53 $95 $46 $96 $106 $96 $14 $58 $440 $53 $47 $97 $82 $153 $22 $2 $61 $149 $31 $57 $51 $26 $70 $3 $40 $299 $38 $136 $50 $23 $170 $3 $61 $85 $14 $41 $0 $440 $11 $65 $174 $60 $112 $13 $8 $208 $143 $14 $159 $166 $24 $28 $71 $44 $70 $35 $30 $15 $12 $19 $145 $9 $46 $90 $81 $1 $3 $22 $94 $36 $18 $134 $312 $42 $20 $25 $70 $1 $15 $175 $125 $61 $148 $25 $27 $106 $5 $88 $23 $104 $9 $35 $48 $39 $88 $109 $13 $19 $37 $16 $40 $123 $13 $15 $22 $39 $52 $57 $43 $76 $14 $59 $19 $20 $145 $2 $90 $33 $57 $29 $242 $18 $17 $18 $192 $46 $85 $29 $114 $80 $6 $67 $15 $158 $3 $14 $17 $379 $12 $292 $14 $2 $20 $31 $33 $333 $20 $100 $32 $69 $5 $29 $16 $331 $81 $20 $24 $66 $55 $72 $66 $124 $29 $25 $60 $33 $32 $22 $13 $27 $83 $9 $6 

In [ ]:
# This is how to save the model if you want to, particularly if you run this on a larger dataset

# import joblib
# joblib.dump(rf_model, "random_forest.joblib")

## Introducing XGBoost

Like Random Forest, XGBoost is also an ensemble model that combines multiple decision trees.

But unlike Random Forest, XGBoost builds one tree after another, with each next tree correcting for errors in the prior trees, using 'gradient descent'.

It's much faster than Random Forest, so we can run it for the full dataset, and it's typically better at generalizing.

**If this import doesn't work, please skip this! It's not required. On a Mac, you might need to do `brew install libomp` in the terminal.**

In [29]:
import xgboost as xgb

In [30]:
np.random.seed(42)

xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X, prices)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [31]:
def xg_boost(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

In [32]:
evaluate(xg_boost, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$104 $68 $26 $3 $166 $186 $9 $39 $84 $199 $527 $209 $117 $68 $13 $30 $39 $28 $1 $10 $9 $17 $30 $78 $48 $191 $23 $23 $69 $23 $132 $99 $30 $2 $106 $187 $3 $20 $161 $112 $135 $40 $10 $15 $140 $38 $46 $12 $24 $75 $12 $33 $232 $34 $73 $140 $91 $125 $35 $14 $79 $11 $73 $29 $305 $68 $39 $142 $41 $107 $17 $64 $148 $139 $43 $71 $207 $3 $60 $55 $76 $60 $25 $23 $46 $16 $12 $137 $53 $105 $79 $110 $28 $1 $68 $91 $15 $22 $129 $302 $65 $64 $27 $75 $13 $31 $47 $142 $45 $68 $3 $36 $191 $36 $6 $4 $137 $1 $61 $65 $65 $117 $32 $24 $112 $18 $4 $134 $73 $15 $31 $23 $15 $14 $75 $78 $116 $27 $38 $54 $19 $70 $30 $101 $52 $69 $51 $287 $14 $4 $11 $95 $30 $67 $33 $103 $25 $25 $11 $1 $167 $20 $29 $19 $511 $9 $163 $41 $22 $7 $35 $1 $312 $38 $31 $49 $13 $30 $59 $131 $368 $122 $70 $2 $85 $84 $70 $35 $65 $50 $38 $46 $17 $22 $6 $47 $12 $26 $22 $17 

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">Traditional ML isn't just useful for learning the history; it's still heavily used in industry today, particularly for tasks where there are clearly identifiable features. It's worth spending time exploring the algorithms and experimenting here. See if you can beat my numbers with Traditional ML! I ran the Random Forest for the entire 800,000 training dataset. It took about 15 hours to run, and it ended up getting a low error of $56.40. Traditional ML can do well - try it for yourself.</span>
        </td>
    </tr>
</table>